In [0]:
-- Player - Stats by Month
-- Month-by-month production.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,                     -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,                -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type,                    -- 2 = regular season, 3 = playoffs
    'Nick Suzuki' AS player,           -- e.g. 'Nick Suzuki'; NULL = every player
    CAST(NULL AS STRING) AS position   -- C, L, R, D, F = all forwards, W = all wingers; NULL = all skaters
),
player_games AS (                        -- one row per skater per game dressed (game rosters)
  SELECT r.game_id, r.player_id, p.full_name, r.position_code, r.sweater_number,
         r.team_id, t.abbrev AS team, o.abbrev AS opponent, r.team_id = g.home_team_id AS is_home,
         g.season, g.game_date, g.home_team_id,
         CASE WHEN IF(r.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game   g ON r.game_id   = g.game_id
  JOIN nhl.gold.dim_player p ON r.player_id = p.player_id
  JOIN nhl.gold.dim_team   t ON r.team_id   = t.team_id
  JOIN nhl.gold.dim_team   o ON o.team_id   = IF(r.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
    AND (pr.player IS NULL OR lower(p.full_name) = lower(pr.player))
    AND r.position_code <> 'G'
    AND (pr.position IS NULL OR r.position_code = pr.position
         OR (pr.position = 'F' AND r.position_code IN ('C', 'L', 'R'))
         OR (pr.position = 'W' AND r.position_code IN ('L', 'R')))
),
events AS (                              -- plays where one of those players had a role
  SELECT pg.player_id AS pid, f.*
  FROM nhl.gold.fact_event f
  JOIN player_games pg
    ON f.game_id = pg.game_id
   AND pg.player_id IN (f.scoring_player_id, f.assist1_player_id, f.assist2_player_id,
                        f.shooting_player_id, f.blocking_player_id, f.hitting_player_id,
                        f.hittee_player_id, f.winning_player_id, f.losing_player_id,
                        f.committed_by_player_id, f.drawn_by_player_id, f.player_id)
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
),
stats AS (
  SELECT pg.full_name AS full_name,
         date_format(pg.game_date, 'yyyy-MM') AS month,
         COUNT(DISTINCT pg.game_id, pg.player_id) AS games_played,
         COUNT_IF(e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid) AS goals,
         COUNT_IF(e.type_desc_key = 'goal' AND e.pid IN (e.assist1_player_id, e.assist2_player_id)) AS assists,
         COUNT_IF((e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid)
               OR (e.type_desc_key = 'shot-on-goal' AND e.shooting_player_id = e.pid)) AS shots,
         COUNT_IF(e.type_desc_key = 'missed-shot' AND e.shooting_player_id = e.pid) AS missed_shots,
         COUNT_IF(e.type_desc_key = 'blocked-shot' AND e.shooting_player_id = e.pid) AS own_shots_blocked,
         COUNT_IF(e.type_desc_key = 'hit' AND e.hitting_player_id = e.pid) AS hits,
         COUNT_IF(e.type_desc_key = 'blocked-shot' AND e.blocking_player_id = e.pid) AS blocked_shots,
         COUNT_IF(e.type_desc_key = 'takeaway' AND e.player_id = e.pid) AS takeaways,
         COUNT_IF(e.type_desc_key = 'giveaway' AND e.player_id = e.pid) AS giveaways,
         SUM(IF(e.type_desc_key = 'penalty' AND e.committed_by_player_id = e.pid,
               COALESCE(e.penalty_duration, 0), 0)) AS pim
  FROM player_games pg
  LEFT JOIN events e ON e.pid = pg.player_id AND e.game_id = pg.game_id
  GROUP BY pg.full_name, date_format(pg.game_date, 'yyyy-MM')
),
final AS (
  SELECT full_name,
         month,
         games_played,
         goals,
         assists,
         goals + assists AS points,
         ROUND((goals + assists) / games_played, 2) AS points_per_game,
         shots,
         ROUND(100.0 * goals / NULLIF(shots, 0), 1) AS shooting_pct,
         ROUND(100.0 * shots / NULLIF((shots + missed_shots + own_shots_blocked), 0), 1) AS shots_on_net_pct,
         hits,
         blocked_shots,
         takeaways,
         giveaways,
         pim
  FROM stats
)
SELECT *
FROM final
ORDER BY full_name, month;